# Fine-tune Vi-Qwen2 — PC Builder + Compatibility
Loads multiple JSONL datasets, combines + shuffles them, then trains and exports GGUF.

In [ ]:
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

**PyTorch sanity check:** run this notebook in a fresh GPU runtime. If the next cell reports a missing `torch._utils`, use **Runtime → Disconnect and delete runtime**, reconnect, run the install cell once, then use **Runtime → Restart session** before continuing. Do not repair Torch piecemeal in the broken runtime.

In [ ]:
import torch
import torch._utils
print('PyTorch OK:', torch.__version__, '| CUDA:', torch.version.cuda)

In [ ]:
# ── Upload one or more JSONL files ──────────────────────────────────────────
# All files MUST use the same {"messages": [...]} chat format.
from google.colab import files
uploaded = files.upload()   # select multiple files (Ctrl+click)
DATA_FILES = list(uploaded.keys())
assert 'pc_builder_adaptive.jsonl' in DATA_FILES, 'Upload the PC Builder decision + state dataset'
print('Uploaded:', DATA_FILES)

MODEL  = 'AITeamVN/Vi-Qwen2-1.5B-RAG'
OUTPUT = '/content/vi-qwen2-pc-builder-state-v2'

In [ ]:
# ── Combine all files into one dataset ──────────────────────────────────────
# Does NOT hurt training quality as long as:
#   1. All files share the same {messages} schema
#   2. We shuffle after combining (prevents task ordering bias)
#   3. No single file dominates (check counts below)
from datasets import load_dataset, concatenate_datasets

parts = []
for path in DATA_FILES:
    ds = load_dataset('json', data_files=f'/content/{path}', split='train')
    print(f'  {path}: {len(ds)} examples')
    if path == 'pc_builder_adaptive.jsonl':
        assert len(ds) == 405, f'Expected 405 PC Builder rows, got {len(ds)}'
    parts.append(ds)

dataset = concatenate_datasets(parts).shuffle(seed=3407)
print(f'\nCombined: {len(dataset)} total examples')

In [ ]:
# ── Load model + LoRA ───────────────────────────────────────────────────────
from unsloth import FastLanguageModel, is_bfloat16_supported
from unsloth.chat_templates import train_on_responses_only
from trl import SFTConfig, SFTTrainer

MAX_SEQ_LENGTH = 8192
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL, max_seq_length=MAX_SEQ_LENGTH, load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=32,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
)

dataset = dataset.map(lambda row: {
    'text': tokenizer.apply_chat_template(
        row['messages'], tokenize=False, add_generation_prompt=False
    )
})
token_lengths = [
    len(tokenizer(row['text'], add_special_tokens=False)['input_ids'])
    for row in dataset
]
assert max(token_lengths) <= MAX_SEQ_LENGTH, (
    f'Training row would be truncated: {max(token_lengths)} > {MAX_SEQ_LENGTH}'
)
assert all('<|im_start|>assistant\n' in row['text'] for row in dataset), (
    'Tokenizer chat template does not match the response-only mask marker'
)
print(f'Max tokenized row: {max(token_lengths)} / {MAX_SEQ_LENGTH}')

In [ ]:
# ── Train ───────────────────────────────────────────────────────────────────
trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        output_dir=OUTPUT,
        dataset_text_field='text',
        max_length=MAX_SEQ_LENGTH,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=1,
        num_train_epochs=3,
        learning_rate=2e-4,
        warmup_steps=10,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        optim='adamw_8bit',
        logging_steps=10,
        seed=3407,
        report_to='none',
    ),
)
trainer = train_on_responses_only(
    trainer,
    instruction_part='<|im_start|>user\n',
    response_part='<|im_start|>assistant\n',
)
supervised_counts = [
    sum(token != -100 for token in row['labels'])
    for row in trainer.train_dataset
]
assert supervised_counts and min(supervised_counts) > 0, (
    'Response-only masking removed every assistant token from at least one row'
)
print(f'Supervised response tokens per row: {min(supervised_counts)}..{max(supervised_counts)}')
trainer.train()
model.save_pretrained(OUTPUT)
tokenizer.save_pretrained(OUTPUT)
model.save_pretrained_gguf(
    f'{OUTPUT}/gguf', tokenizer, quantization_method='q8_0'
)

Download the `.gguf` from `/content/vi-qwen2-pc-builder-state-v2/gguf` in Colab's Files panel and import it under a new Ollama tag. Keep the existing Q8 model until the live acceptance benchmark passes.